
Modify Jupyter notebook simple_model_with_data.ipynb with two GPU in Kaggle (GPU T4 x2)

References: https://docs.pytorch.org/tutorials/intermediate/ddp_tutorial.html


In [ ]:
%%writefile simple_model_with_data_multigpu.py

import torch
import torch.nn as nn
import os

from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
from torchvision import datasets, transforms


# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

# add device, rank in the parameters for train/test
def train(dataloader, model, loss_fn, optimizer, device, rank):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0 and rank == 0: # print only on rank 0!
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn, device, rank):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    if rank == 0:
        print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

#def ddp_setup(rank, world_size):
#    os.environ['MASTER_ADDR'] = 'localhost'
#    os.environ['MASTER_PORT'] = '12355'
#    torch.cuda.set_device(rank)
#    init_process_group(backend="nccl", rank=rank, world_size=world_size)

# ddp training
def kaggle_ddp_train(gpu_id, world_size):

    # Manual setup like torchrun.py!

    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    init_process_group(backend='nccl', rank=gpu_id, world_size=torch.cuda.device_count())
    torch.cuda.set_device(gpu_id)
    device = torch.device(f'cuda:{gpu_id}')

    # Load Data inside the process
    training_data = datasets.FashionMNIST(root="data", train=True, download=False, transform=transforms.ToTensor())
    test_data = datasets.FashionMNIST(root="data", train=False, download=False, transform=transforms.ToTensor())

    model = NeuralNetwork().to(device)
    model = DDP(model, device_ids=[gpu_id])

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

    train_sampler = DistributedSampler(training_data, num_replicas=world_size, rank=gpu_id)
    train_dataloader = DataLoader(training_data, batch_size=64, sampler=train_sampler)

    test_sampler = DistributedSampler(test_data, num_replicas=world_size, rank=gpu_id)
    test_dataloader = DataLoader(test_data, batch_size=64, sampler=test_sampler)

    epochs = 5
    for t in range(epochs):
        if gpu_id == 0:
            print(f"Epoch {t+1}\n--------------------------------------------------------------")
        train_sampler.set_epoch(t)
        train(train_dataloader, model, loss_fn, optimizer, device, gpu_id)
        test(test_dataloader, model, loss_fn, device, gpu_id)

    if gpu_id == 0:
        torch.save(model.module.state_dict(), "model.pth")
        print("Training Complete!")

    destroy_process_group()

Writing simple_model_with_data_multigpu.py


In [ ]:
import torch
import torch.multiprocessing as mp
from torchvision import datasets, transforms
from simple_model_with_data_multigpu import kaggle_ddp_train

# download the data outside for child process in case
datasets.FashionMNIST(root="data", train=True, download=True)
datasets.FashionMNIST(root="data", train=False, download=True)

if __name__ == "__main__":
    world_size = torch.cuda.device_count()
    print(f"spawn-ing {world_size} processes...")

    #  the datasets are loaded inside the script
    mp.spawn(kaggle_ddp_train, args=(world_size,), nprocs=world_size)

100%|██████████| 26.4M/26.4M [00:01<00:00, 15.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 271kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.01MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.1MB/s]

spawn-ing 2 processes...


Epoch 1
--------------------------------------------------------------
loss: 2.307858  [   64/60000]
loss: 2.303017  [ 6464/60000]
loss: 2.290653  [12864/60000]
loss: 2.268379  [19264/60000]
loss: 2.245694  [25664/60000]
Test Error: 
 Accuracy: 20.0%, Avg loss: 2.230414 

Epoch 2
--------------------------------------------------------------
loss: 2.232253  [   64/60000]
loss: 2.225557  [ 6464/60000]
loss: 2.223803  [12864/60000]
loss: 2.200866  [19264/60000]
loss: 2.133036  [25664/60000]
Test Error: 
 Accuracy: 22.8%, Avg loss: 2.147409 

Epoch 3
--------------------------------------------------------------
loss: 2.175798  [   64/60000]
loss: 2.142723  [ 6464/60000]
loss: 2.105926  [12864/60000]
loss: 2.048674  [19264/60000]
loss: 2.038984  [25664/60000]
Test Error: 
 Accuracy: 25.9%, Avg loss: 2.036428 

Epoch 4
--------------------------------------------------------------
loss: 2.057904  [   64/60000]
loss: 2.013217  [ 6464/60000]
loss: 1.990530  [12864/60000]
loss: 1.902270  [192

In [ ]:
import torch
from torchvision import datasets, transforms
from simple_model_with_data_multigpu import NeuralNetwork

# 1. Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device) # better be cuda as selecting GPU T4x2

# Load the Model
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

cuda


<All keys matched successfully>

In [ ]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

# re-load for testing!
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=False,
    transform=transforms.ToTensor(),
)

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
